# OjoVial — Prueba de Cámara
Detección en tiempo real con webcam. Presiona **q** para salir.

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import time

In [2]:
CLASSES = {
    0: ("Pare (Stop)",             (68, 68, 255)),
    1: ("Ceda el Paso (Yield)",    (0, 215, 255)),
    2: ("No Right Turn",           (0, 140, 255)),
    3: ("No U-Turn",               (255, 136, 68)),
    4: ("Go Straight",             (68, 255, 68)),
    5: ("Speed Limit 40",          (187, 187, 187)),
    6: ("Speed Limit 60",          (153, 153, 153)),
    7: ("No Parking",              (204, 51, 153)),
}

CONF_THRESHOLD = 0.80
IOU_THRESHOLD  = 0.50
MODEL_PATH     = "models/best.pt"
IMG_SIZE       = 640

In [3]:
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {model.names}")

Model loaded: runs/detect/runs/traffic_signs/yolov8s_finetuned_v5/weights/best.pt
Classes: {0: 'pare', 1: 'ceda_el_paso', 2: 'prohibido_girar_derecha', 3: 'prohibido_girar_izquierda', 4: 'siga_de_frente', 5: 'velocidad_maxima_40', 6: 'velocidad_maxima_60', 7: 'prohibido_estacionar'}


In [5]:
def on_trackbar(val):
    pass


def draw_detections(frame, results):
    for r in results:
        boxes  = r.boxes
        if boxes is None:
            continue
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf  = float(box.conf[0])
            clsid = int(box.cls[0])

            name, color = CLASSES.get(clsid, (f"Class {clsid}", (255, 255, 255)))

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = f"{name} {conf * 100:.1f}%"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)
            cv2.putText(frame, label, (x1 + 3, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    return frame

cap = cv2.VideoCapture(1)
if not cap.isOpened():
    print("ERROR: No se pudo abrir la cámara")
else:
    window = "OjoVial - Prueba de Cámara"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window, 960, 720)

    cv2.createTrackbar("Conf %", window, 65, 95, on_trackbar)
    cv2.createTrackbar("IoU %",  window, 50, 90, on_trackbar)

    print("Cámara abierta. Presiona 'q' to quit.")

    fps_time = time.time()
    fps_count = 0
    fps_display = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error al leer frame")
            break

        conf = cv2.getTrackbarPos("Conf %", window) / 100.0
        iou  = cv2.getTrackbarPos("IoU %",  window) / 100.0

        results = model.predict(
            source=frame,
            conf=conf,
            iou=iou,
            imgsz=IMG_SIZE,
            verbose=False,
        )

        frame = draw_detections(frame, results)

        fps_count += 1
        if time.time() - fps_time >= 1.0:
            fps_display = fps_count
            fps_count = 0
            fps_time = time.time()

        cv2.putText(frame, f"FPS: {fps_display}  Conf: {conf:.2f}  IoU: {iou:.2f}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.imshow(window, frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Cámara liberada.")

Camera opened. Press 'q' to quit.
Camera released.
